## 11_v4_model_run
  Runs radiation and normalized exponential gravity (unconstrained, expected matrix) for HU and
  SK LAU units and aggregates to NUTS3. No observed matrix exists for HU/SK, so this is a model
  output, not a validation. Output: data/processed/11_v4_model_run/. Environment: nuts3_tdlm.

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
SK_FILE_REL = "data/raw/commuting/sk/SLO2021IV.csv"
SK_SEP = ";"
SK_ENCODING = "cp1250"
# lowercase stems searched in the column names (work, commuting, abroad)
KEYWORDS = ["prác", "chádz", "zahrani", "dochád", "odchád"]

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

sk_file = ROOT / SK_FILE_REL
print("Root found; file exists:", sk_file.exists())  # no absolute path printed

try:
    # read only the header row; quoted multi-line headers are handled by the parser
    header = pd.read_csv(sk_file, sep=SK_SEP, encoding=SK_ENCODING, nrows=0)
except Exception as exc:
    # print only the exception class name (no paths in the output)
    raise RuntimeError("Read failed: " + type(exc).__name__) from None

print("Total columns:", len(header.columns))

hits = []
for i, col in enumerate(header.columns):
    # collapse line breaks only for display; the data itself is untouched
    shown = " ".join(str(col).split())
    if any(k in shown.lower() for k in KEYWORDS):
        hits.append((i, shown))

print("Matching columns:", len(hits))
for i, shown in hits:
    print(i, "|", shown)

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
CAL_REL = "data/processed/09_lenormand_multilevel/calibrated_exponents_levels.csv"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

try:
    cal = pd.read_csv(ROOT / CAL_REL)
except Exception as exc:
    # print only the exception class name (no paths in the output)
    raise RuntimeError("Read failed: " + type(exc).__name__) from None

print("Rows, columns:", cal.shape)
print("Columns:", list(cal.columns))

# unique values of the text columns (small sets only)
for col in cal.columns:
    if cal[col].dtype == object:
        vals = cal[col].unique()
        print(col, "->", list(vals)[:20])

print(cal.head(3).to_string())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- config (all names and paths in one place) ----
CAL_REL = "data/processed/09_lenormand_multilevel/calibrated_exponents_levels.csv"
OUT_REL = "data/processed/11_v4_model_run/beta_regression.csv"
LAW = "NGravExp"
X_COL = "mean_unit_area_km2"
EXP_COL = "Exponent"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

cal = pd.read_csv(ROOT / CAL_REL)
d = cal[cal["Law"] == LAW]

# power-law fit over all points of the law = straight line in log-log space
lx, ly = np.log10(d[X_COL]), np.log10(d[EXP_COL])
slope, intercept = np.polyfit(lx, ly, 1)
r2 = np.corrcoef(lx, ly)[0, 1] ** 2

print("Law:", LAW, "| points:", len(d),
      "| levels:", d.groupby(["country", "level"]).ngroups)
print("slope:", round(slope, 4), "| prefactor:", round(10 ** intercept, 5),
      "| R2:", round(r2, 3))

out = pd.DataFrame([{
    "law": LAW, "n_points": len(d), "slope": slope,
    "intercept_log10": intercept, "prefactor": 10 ** intercept, "r2": r2,
}])
(ROOT / OUT_REL).parent.mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / OUT_REL, index=False)
print("Saved:", OUT_REL)

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
UNITS_DIR_REL = "data/interim/10_v4_inputs"
UNITS_PATTERN = "hu_sk_lau_units*.parquet"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

matches = sorted((ROOT / UNITS_DIR_REL).glob(UNITS_PATTERN))
print("Matching files:", [m.relative_to(ROOT).as_posix() for m in matches])
if len(matches) != 1:
    raise RuntimeError("Expected exactly one units file") from None

units = pd.read_parquet(matches[0])
print("Rows, columns:", units.shape)
print("Columns:", list(units.columns))
print(units.head(3).to_string())

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
UNITS_REL = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
REG_REL = "data/processed/11_v4_model_run/beta_regression.csv"
OUT_REL = "data/processed/11_v4_model_run/beta_hu_sk.csv"
AREA_COL = "AREA_KM2"   # same column as used for <S> in notebook 09
COUNTRY_COL = "CNTR_CODE"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

units = pd.read_parquet(ROOT / UNITS_REL)
reg = pd.read_csv(ROOT / REG_REL).iloc[0]

rows = []
groups = [(c, units[units[COUNTRY_COL] == c]) for c in sorted(units[COUNTRY_COL].unique())]
groups.append(("HU+SK", units))
for name, g in groups:
    s_mean = g[AREA_COL].mean()
    # beta from the power-law regression: prefactor * <S> ** slope
    beta = reg["prefactor"] * s_mean ** reg["slope"]
    rows.append({"area_col": AREA_COL, "group": name, "n_units": len(g),
                 "mean_unit_area_km2": s_mean, "beta": beta})

out = pd.DataFrame(rows)
print(out.round(4).to_string(index=False))

(ROOT / OUT_REL).parent.mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / OUT_REL, index=False)
print("Saved:", OUT_REL)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

# ---- config (all names and paths in one place) ----
UNITS_REL = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
DIST_REL = "data/interim/11_v4_model_run/distance_km.npy"
ORDER_REL = "data/processed/11_v4_model_run/units_order.csv"
X_COL, Y_COL = "x_3035", "y_3035"          # metres, EPSG:3035
KEEP_COLS = ["CNTR_CODE", "LAU_ID", "LAU_NAME", "POP_2021", "AREA_KM2"]

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

units = pd.read_parquet(ROOT / UNITS_REL).reset_index(drop=True)
n = len(units)
print("Units:", n, "| matrix memory (float64):", round(n * n * 8 / 1e6), "MB")

# save the row order so that the matrix rows can always be matched to the units
order_file = ROOT / ORDER_REL
order_file.parent.mkdir(parents=True, exist_ok=True)
if not order_file.exists():
    units[KEEP_COLS].to_csv(order_file, index_label="row")
    print("Saved:", ORDER_REL)
else:
    print("Exists, skipped:", ORDER_REL)

dist_file = ROOT / DIST_REL
dist_file.parent.mkdir(parents=True, exist_ok=True)
if dist_file.exists():
    print("Exists, skipped:", DIST_REL)
    dist = np.load(dist_file)
else:
    # same as notebook 09: Euclidean distance of the centroids in metres, divided by 1000
    dist = squareform(pdist(units[[X_COL, Y_COL]].to_numpy())) / 1000
    np.save(dist_file, dist)
    print("Saved:", DIST_REL)

# sanity checks: shape, off-diagonal minimum, maximum, coincident centroids
off = dist[~np.eye(n, dtype=bool)]
print("Shape:", dist.shape)
print("Min off-diagonal (km):", round(float(off.min()), 4),
      "| max (km):", round(float(off.max()), 1))
print("Different unit pairs with zero distance:", int((off == 0).sum() // 2))
del off

In [ ]:
import numpy as np
from TDLM import tdlm

# ---- config (toy example only) ----
MASS = np.array([1000.0, 2000.0, 3000.0, 4000.0])
# distances in km between 4 units on a line (symmetric, zero diagonal)
COORD = np.array([0.0, 10.0, 25.0, 60.0])
DIST = np.abs(COORD[:, None] - COORD[None, :])
OUT_TRIPS = np.array([10.0, 20.0, 30.0, 40.0])   # sum = 100
LAWS = [("Rad", 1.0), ("NGravExp", 0.05)]

cases = {
    "no out_trips": dict(),
    "out_trips (sum 100)": dict(out_trips=OUT_TRIPS),
    "out_trips x2 (sum 200)": dict(out_trips=2 * OUT_TRIPS),
}

for law, expo in LAWS:
    for name, kw in cases.items():
        try:
            sim = tdlm.run_law_model(
                law=law, mass_origin=MASS, mass_destination=MASS, distance=DIST,
                exponent=np.array([expo]), model="UM", average=True,
                verbose=False, **kw)
            # single exponent -> array (1, n, n); several -> dict
            mat = np.asarray(sim if not isinstance(sim, dict) else next(iter(sim.values())))
            print(law, "|", name, "| shape:", mat.shape,
                  "| sum:", round(float(mat.sum()), 3),
                  "| diagonal sum:", round(float(np.trace(mat[0])), 3))
        except Exception as exc:
            # print only the exception class name (no paths in the output)
            print(law, "|", name, "| failed:", type(exc).__name__)

In [ ]:
import numpy as np
from TDLM import tdlm

# ---- config (toy example only) ----
MASS = np.array([1000.0, 2000.0, 3000.0, 4000.0])
COORD = np.array([0.0, 10.0, 25.0, 60.0])           # km on a line
DIST = np.abs(COORD[:, None] - COORD[None, :])
OUT_A = np.array([25.0, 25.0, 25.0, 25.0])          # sum = 100
OUT_B = np.array([100.0, 0.0, 0.0, 0.0])            # sum = 100, concentrated
LAWS = [("Rad", 1.0), ("NGravExp", 0.05)]

def run(law, expo, out_trips):
    sim = tdlm.run_law_model(
        law=law, mass_origin=MASS, mass_destination=MASS, distance=DIST,
        exponent=np.array([expo]), model="UM", average=True,
        out_trips=out_trips, verbose=False)
    return np.asarray(sim if not isinstance(sim, dict) else next(iter(sim.values())))[0]

for law, expo in LAWS:
    try:
        a = run(law, expo, OUT_A)
        b = run(law, expo, OUT_B)
        print(law, "| max abs difference:", float(np.abs(a - b).max()),
              "| sums:", round(float(a.sum()), 3), round(float(b.sum()), 3))
    except Exception as exc:
        # print only the exception class name (no paths in the output)
        print(law, "| failed:", type(exc).__name__)

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
NC_REL = "data/processed/10_v4_inputs/nc_summary.csv"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

try:
    nc = pd.read_csv(ROOT / NC_REL)
except Exception as exc:
    # print only the exception class name (no paths in the output)
    raise RuntimeError("Read failed: " + type(exc).__name__) from None

print("Columns:", list(nc.columns))
print(nc.to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- config (all names and paths in one place) ----
ORDER_REL = "data/processed/11_v4_model_run/units_order.csv"
DIST_REL = "data/interim/11_v4_model_run/distance_km.npy"
NC_REL = "data/processed/10_v4_inputs/nc_summary.csv"
OUT_REL = "data/processed/11_v4_model_run/model_inputs_summary.csv"
COUNTRY_COL, POP_COL = "CNTR_CODE", "POP_2021"
NC_COL = "nc_high"          # HU: single value, SK: upper value (decision 2026-10-19)
COUNTRIES = ["HU", "SK"]

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

order = pd.read_csv(ROOT / ORDER_REL, dtype={"LAU_ID": str})
nc = pd.read_csv(ROOT / NC_REL).set_index("country")
# memory map: the full 6044 x 6044 matrix is not loaded, only the needed blocks
dist_all = np.load(ROOT / DIST_REL, mmap_mode="r")
assert dist_all.shape == (len(order), len(order)), "matrix and unit order do not match"

def build_inputs(country):
    """Mass, distance block and out_trips of one country (rows = positions in the saved matrix)."""
    idx = order.index[order[COUNTRY_COL] == country].to_numpy()
    mass = order.loc[idx, POP_COL].to_numpy(dtype=float)
    dist = np.asarray(dist_all[np.ix_(idx, idx)])
    n_c = float(nc.loc[country, NC_COL])
    # UM uses only the sum of out_trips (toy test in section 7); spread N_c by population
    out_trips = n_c * mass / mass.sum()
    return idx, mass, dist, out_trips, n_c

rows = []
for country in COUNTRIES:
    idx, mass, dist, out_trips, n_c = build_inputs(country)
    rows.append({
        "country": country, "n_units": len(idx), "N_c": n_c,
        "out_trips_sum": float(out_trips.sum()),
        "dist_shape": f"{dist.shape[0]}x{dist.shape[1]}",
        "dist_max_km": float(dist.max()),
        "diag_zero": bool(np.all(np.diag(dist) == 0)),
        "symmetric": bool(np.allclose(dist, dist.T)),
        "min_population": float(mass.min()),
    })
    del idx, mass, dist, out_trips

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

out_file = ROOT / OUT_REL
summary.to_csv(out_file, index=False)
print("Saved:", OUT_REL)

In [ ]:
import time
import numpy as np
import pandas as pd
from TDLM import tdlm

# ---- config (all names and paths in one place) ----
BETA_REL = "data/processed/11_v4_model_run/beta_hu_sk.csv"
MAT_DIR_REL = "data/interim/11_v4_model_run"
SUMMARY_REL = "data/processed/11_v4_model_run/model_runs_summary.csv"
LAWS = ["Rad", "NGravExp"]
RAD_EXPONENT = 1.0                 # Rad has no fitted parameter
# COUNTRIES, ROOT and build_inputs() come from the previous cell (section 9)

beta_tab = pd.read_csv(ROOT / BETA_REL).set_index("group")
(ROOT / MAT_DIR_REL).mkdir(parents=True, exist_ok=True)

rows = []
for country in COUNTRIES:
    idx, mass, dist, out_trips, n_c = build_inputs(country)
    for law in LAWS:
        rel = f"{MAT_DIR_REL}/{country}_{law}_UM_expected.npy"
        expo = float(beta_tab.loc[country, "beta"]) if law == "NGravExp" else RAD_EXPONENT
        t0 = time.time()
        if (ROOT / rel).exists():
            mat = np.load(ROOT / rel).astype(float)
            status = "loaded"
        else:
            try:
                sim = tdlm.run_law_model(
                    law=law, mass_origin=mass, mass_destination=mass, distance=dist,
                    exponent=np.array([expo]), model="UM", out_trips=out_trips,
                    average=True, verbose=False)
            except Exception as exc:
                # print only the exception class name (no paths in the output)
                print(country, law, "| failed:", type(exc).__name__)
                continue
            mat = np.asarray(sim if not isinstance(sim, dict) else next(iter(sim.values())))[0]
            np.save(ROOT / rel, mat.astype(np.float32))   # checkpoint per run
            status = "computed"
        rows.append({"country": country, "law": law, "status": status, "n_units": len(mass),
                     "exponent": expo, "N_c": n_c, "matrix_sum": float(mat.sum()),
                     "diagonal_sum": float(np.trace(mat)), "max_flow": float(mat.max()),
                     "seconds": round(time.time() - t0, 1)})
        print(rows[-1])
        del mat
    del idx, mass, dist, out_trips

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv(ROOT / SUMMARY_REL, index=False)
print("Saved:", SUMMARY_REL)

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
MAP_REL = "data/raw/eurostat/EU-27-LAU-2021-NUTS-2021.xlsx"
PREVIEW_ROWS = 3

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

try:
    xls = pd.ExcelFile(ROOT / MAP_REL)
except Exception as exc:
    # print only the exception class name (no paths in the output)
    raise RuntimeError("Open failed: " + type(exc).__name__) from None

print("Sheets:", xls.sheet_names)
for sheet in xls.sheet_names:
    # read every column as text so that leading zeros are visible
    df = xls.parse(sheet, dtype=str)
    print("\n== sheet:", sheet, "| rows:", len(df))
    print("Columns:", list(df.columns))
    print(df.head(PREVIEW_ROWS).to_string())

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
MAP_REL = "data/raw/eurostat/EU-27-LAU-2021-NUTS-2021.xlsx"
UNITS_REL = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
COUNTRIES = ["HU", "SK"]                 # sheet names equal the country codes
SHEET_LAU, SHEET_NUTS = "LAU CODE", "NUTS 3 CODE"
UNITS_COUNTRY, UNITS_ID = "CNTR_CODE", "LAU_ID"
MAX_SHOWN = 10
CUT = 60                                 # shown length of merged_lau_ids

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

units = pd.read_parquet(ROOT / UNITS_REL)
units[UNITS_ID] = units[UNITS_ID].astype(str)

for c in COUNTRIES:
    # read as text so that leading zeros are kept
    m = pd.read_excel(ROOT / MAP_REL, sheet_name=c, dtype=str)[[SHEET_LAU, SHEET_NUTS]]
    u = units[units[UNITS_COUNTRY] == c][[UNITS_ID, "n_merged_parts", "merged_lau_ids"]]
    j = u.merge(m, left_on=UNITS_ID, right_on=SHEET_LAU, how="left")
    j["merged_lau_ids"] = j["merged_lau_ids"].astype(str).str[:CUT]
    unmatched = j[j[SHEET_NUTS].isna()]
    print(f"{c} | units: {len(u)} | sheet rows: {len(m)} | duplicate LAU codes in sheet: "
          f"{int(m[SHEET_LAU].duplicated().sum())} | unmatched units: {len(unmatched)} | "
          f"NUTS3 regions: {j[SHEET_NUTS].nunique()}")
    print("Unmatched (first rows):")
    print(unmatched.head(MAX_SHOWN).to_string(index=False))
    print("Merged units (n_merged_parts > 1):")
    print(j[j["n_merged_parts"] > 1].to_string(index=False))
    print()

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
MAP_REL = "data/raw/eurostat/EU-27-LAU-2021-NUTS-2021.xlsx"
UNITS_REL = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
ORDER_REL = "data/processed/11_v4_model_run/units_order.csv"
OUT_REL = "data/processed/11_v4_model_run/lau_to_nuts3.csv"
COUNTRIES = ["HU", "SK"]                 # sheet names equal the country codes
SHEET_LAU, SHEET_NUTS = "LAU CODE", "NUTS 3 CODE"
COUNTRY_COL, ID_COL, PARTS_COL = "CNTR_CODE", "LAU_ID", "merged_lau_ids"
PARTS_SEP = ";"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

units = pd.read_parquet(ROOT / UNITS_REL).reset_index(drop=True)
order = pd.read_csv(ROOT / ORDER_REL, dtype={ID_COL: str})
# the saved matrix order must be identical to the order used here
assert (units[ID_COL].astype(str).to_numpy() == order[ID_COL].to_numpy()).all(), "unit order differs"

# LAU code -> NUTS3 code dictionaries (read as text so that leading zeros are kept)
lookup = {}
for c in COUNTRIES:
    m = pd.read_excel(ROOT / MAP_REL, sheet_name=c, dtype=str)[[SHEET_LAU, SHEET_NUTS]]
    lookup[c] = dict(zip(m[SHEET_LAU], m[SHEET_NUTS]))

rows = []
for r in units.itertuples(index=False):
    country, parts = getattr(r, COUNTRY_COL), str(getattr(r, PARTS_COL)).split(PARTS_SEP)
    codes = {lookup[country].get(x) for x in parts}
    rows.append({"row": len(rows), COUNTRY_COL: country, ID_COL: getattr(r, ID_COL),
                 "n_parts": len(parts),
                 "n_nuts3_found": len(codes - {None}),
                 "n_parts_missing": sum(lookup[country].get(x) is None for x in parts),
                 "NUTS3": next(iter(codes - {None})) if len(codes - {None}) == 1 else None})
out = pd.DataFrame(rows)

for c in COUNTRIES:
    g = out[out[COUNTRY_COL] == c]
    per_region = g.groupby("NUTS3").size()
    print(f"{c} | units: {len(g)} | units with a missing part: {int((g['n_parts_missing'] > 0).sum())}"
          f" | units in more than one NUTS3: {int((g['n_nuts3_found'] > 1).sum())}"
          f" | units without NUTS3: {int(g['NUTS3'].isna().sum())}"
          f" | NUTS3 regions: {per_region.shape[0]}"
          f" | units per region min/max: {int(per_region.min())}/{int(per_region.max())}")

(ROOT / OUT_REL).parent.mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / OUT_REL, index=False)
print("Saved:", OUT_REL)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- config (all names and paths in one place) ----
MAP_REL = "data/processed/11_v4_model_run/lau_to_nuts3.csv"
MAT_DIR_REL = "data/interim/11_v4_model_run"
OUT_REL = "data/processed/11_v4_model_run/nuts3_flows_offdiag.csv"
SUMMARY_REL = "data/processed/11_v4_model_run/nuts3_aggregation_summary.csv"
COUNTRIES = ["HU", "SK"]
LAWS = ["Rad", "NGravExp"]
COUNTRY_COL, NUTS_COL = "CNTR_CODE", "NUTS3"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

out_file, summary_file = ROOT / OUT_REL, ROOT / SUMMARY_REL
if out_file.exists() and summary_file.exists():
    print("Exists, skipped:", OUT_REL, "and", SUMMARY_REL)
else:
    mapping = pd.read_csv(ROOT / MAP_REL, dtype=str)
    mapping["row"] = mapping["row"].astype(int)
    frames, rows = [], []
    for country in COUNTRIES:
        # unit order of the country matrix = order of rows in the saved mapping
        g = mapping[mapping[COUNTRY_COL] == country].sort_values("row")
        regions = sorted(g[NUTS_COL].unique())
        region_pos = {r: i for i, r in enumerate(regions)}
        # P: units x regions indicator matrix, so that M_nuts = P.T @ M @ P
        P = np.zeros((len(g), len(regions)))
        P[np.arange(len(g)), g[NUTS_COL].map(region_pos).to_numpy()] = 1.0
        for law in LAWS:
            mat = np.load(ROOT / f"{MAT_DIR_REL}/{country}_{law}_UM_expected.npy").astype(float)
            assert mat.shape[0] == len(g), "matrix and mapping do not match"
            agg = P.T @ mat @ P
            lau_total = float(agg.sum())          # equals N_c of the LAU run
            within = float(np.trace(agg))         # flow inside the same NUTS3 region
            np.fill_diagonal(agg, 0.0)            # same definition as CZ kraj / PL NUTS3
            between = float(agg.sum())
            rows.append({"country": country, "law": law, "n_regions": len(regions),
                         "lau_total": lau_total, "within_region": within,
                         "within_share": within / lau_total, "between_regions": between})
            print(rows[-1])
            frames.append(pd.DataFrame({
                "country": country, "law": law,
                "origin_nuts3": np.repeat(regions, len(regions)),
                "destination_nuts3": np.tile(regions, len(regions)),
                "flow": agg.ravel()}))
            del mat, agg
    out_file.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(frames, ignore_index=True).to_csv(out_file, index=False)
    pd.DataFrame(rows).to_csv(summary_file, index=False)
    print("Saved:", OUT_REL, "and", SUMMARY_REL)

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
SK_REL = "data/raw/commuting/sk/SLO2021III.csv"
SK_SEP, SK_ENCODING = ";", "cp1250"
# lowercase stems of the columns shown in the preview (unit name / level / outflow)
SHOW_KEYS = ["názov", "kraj", "okres", "úrov", "odchád"]
N_PREVIEW = 15

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

try:
    # read everything as text: the thousands separator is a space, converted later
    sk3 = pd.read_csv(ROOT / SK_REL, sep=SK_SEP, encoding=SK_ENCODING, dtype=str)
except Exception as exc:
    # print only the exception class name (no paths in the output)
    raise RuntimeError("Read failed: " + type(exc).__name__) from None

# collapse line breaks in the header for display only; the data is untouched
shown = {c: " ".join(str(c).split()) for c in sk3.columns}
print("Rows, columns:", sk3.shape)
for i, c in enumerate(sk3.columns):
    print(i, "|", shown[c])

keep = [c for c in sk3.columns if any(k in shown[c].lower() for k in SHOW_KEYS)]
print("\nPreview of the matching columns:")
print(sk3[keep].rename(columns=shown).head(N_PREVIEW).to_string())

In [ ]:
from pathlib import Path
import pandas as pd

# ---- config (all names and paths in one place) ----
SK_REL = "data/raw/commuting/sk/SLO2021III.csv"
FLOWS_REL = "data/processed/11_v4_model_run/nuts3_flows_offdiag.csv"
OUT_REL = "data/processed/11_v4_model_run/sk_kraj_check.csv"
SK_SEP, SK_ENCODING = ";", "cp1250"
COL_AREA, COL_LEVEL = "Územie", "Typ územnej úrovne"
COL_OUT, COL_KRAJ = "Obyvateľstvo (SP) odchádzajúce za prácou", "Kód kraja"
LEVEL_KRAJ = "kraj"
COUNTRY = "SK"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

raw = pd.read_csv(ROOT / SK_REL, sep=SK_SEP, encoding=SK_ENCODING, dtype=str)
# header cells contain line breaks: collapse them (names themselves stay unchanged)
raw.columns = [" ".join(str(c).split()) for c in raw.columns]

def to_number(s):
    # thousands separator is a space (also non-breaking): remove all whitespace
    return pd.to_numeric(s.str.replace(r"\s+", "", regex=True), errors="coerce")

raw["value"] = to_number(raw[COL_OUT])
kraj = raw[raw[COL_LEVEL] == LEVEL_KRAJ][[COL_AREA, COL_KRAJ, "value"]].copy()
sr = raw[raw[COL_KRAJ] == "x"]["value"]          # country-level row(s)
print("Kraj rows:", len(kraj), "| sum of kraj outflows:", int(kraj["value"].sum()))
print("Country-level row value:", sr.tolist())

flows = pd.read_csv(ROOT / FLOWS_REL)
flows = flows[flows["country"] == COUNTRY]
rows = []
for law, g in flows.groupby("law"):
    out_by_region = g.groupby("origin_nuts3")["flow"].sum().sort_index()
    print(f"\nModel {law} | between-region total: {g['flow'].sum():.0f}")
    print(out_by_region.round(0).to_string())
    for region, v in out_by_region.items():
        rows.append({"law": law, "origin_nuts3": region, "model_outflow": v})

print("\nSource kraj rows (order as in the file):")
print(kraj.to_string(index=False))

# save the model-side regional outflows and the source kraj values (small tables)
out = pd.DataFrame(rows)
out["source_kraj_total"] = float(kraj["value"].sum())
out["source_country_row"] = float(sr.iloc[0]) if len(sr) else float("nan")
(ROOT / OUT_REL).parent.mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / OUT_REL, index=False)
print("Saved:", OUT_REL)

In [ ]:
import json
from pathlib import Path
import networkx as nx
import pandas as pd

# ---- config (all names and paths in one place) ----
FLOWS_REL = "data/processed/11_v4_model_run/nuts3_flows_offdiag.csv"
MAP_REL = "data/processed/11_v4_model_run/lau_to_nuts3.csv"
ORDER_REL = "data/processed/11_v4_model_run/units_order.csv"
NAMES_REL = "data/raw/nuts3/NUTS_RG_03M_2024_4326.geojson"
OUT_DIR_REL = "data/processed/11_v4_model_run/graphs"
SUMMARY_REL = "data/processed/11_v4_model_run/graphs/graph_summary.csv"
COUNTRIES = ["HU", "SK"]
LAWS = ["Rad", "NGravExp"]
COUNTRY_COL, NUTS_COL, POP_COL = "CNTR_CODE", "NUTS3", "POP_2021"

# find the project root: walk up from the working directory until a "data" folder exists
ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Project root with a data folder not found") from None

flows = pd.read_csv(ROOT / FLOWS_REL)
mapping = pd.read_csv(ROOT / MAP_REL, dtype=str)
order = pd.read_csv(ROOT / ORDER_REL, dtype=str)
for df in (mapping, order):
    df["row"] = df["row"].astype(int)

# population per NUTS3 region = sum of the LAU unit populations (same row order in both files)
units = mapping[["row", COUNTRY_COL, NUTS_COL]].merge(order[["row", POP_COL]], on="row")
units[POP_COL] = units[POP_COL].astype(float)
region_pop = units.groupby([COUNTRY_COL, NUTS_COL])[POP_COL].sum()

# region names from the GISCO NUTS file; plain json, no geometry library needed
names = {}
try:
    with open(ROOT / NAMES_REL, encoding="utf-8") as f:
        for feat in json.load(f)["features"]:
            props = feat["properties"]
            names[props["NUTS_ID"]] = props["NUTS_NAME"]
except Exception as exc:
    # print only the exception class name (no paths in the output)
    print("Names file not read:", type(exc).__name__, "- names stay empty")

out_dir = ROOT / OUT_DIR_REL
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
for country in COUNTRIES:
    for law in LAWS:
        rel = f"{OUT_DIR_REL}/{country}_{law}_UM_nuts3.graphml"
        g_flows = flows[(flows["country"] == country) & (flows["law"] == law)]
        regions = sorted(set(g_flows["origin_nuts3"]) | set(g_flows["destination_nuts3"]))
        if (ROOT / rel).exists():
            print("Exists, skipped:", rel)
            continue
        G = nx.DiGraph(country=country, law=law, model="UM")
        for r in regions:
            G.add_node(r, id=r, name=names.get(r, ""), country=country,
                       population=float(region_pop.loc[(country, r)]))
        for row in g_flows[g_flows["flow"] > 0].itertuples(index=False):
            G.add_edge(row.origin_nuts3, row.destination_nuts3, weight=float(row.flow))
        nx.write_graphml(G, ROOT / rel)
        rows.append({"country": country, "law": law, "nodes": G.number_of_nodes(),
                     "edges": G.number_of_edges(),
                     "total_weight": sum(d["weight"] for _, _, d in G.edges(data=True)),
                     "nodes_without_name": sum(1 for _, d in G.nodes(data=True) if not d["name"])})
        print(rows[-1])

if rows:
    summary = pd.DataFrame(rows)
    summary.to_csv(ROOT / SUMMARY_REL, index=False)
    print("Saved:", SUMMARY_REL)